# PureEnhance AI — model converter (v4, no upload needed)
Runtime → **Run all**. Models khud download hote hain. Aakhir mein Google Drive ki permission maangi jayegi
(Allow karein); `.onnx` files Drive ke folder **PureEnhance_models** mein save hongi.

In [ ]:
!pip -q install onnx onnxruntime
print('install ok')

In [ ]:
!wget -q -nc https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth
!wget -q -nc https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth
import os
sr_path = 'realesr-general-x4v3.pth'
face_path = 'GFPGANv1.4.pth'
for f in (sr_path, face_path):
    print(f, round(os.path.getsize(f) / 1e6, 1), 'MB')
assert os.path.getsize(sr_path) > 1e6, 'Real-ESRGAN download fail'
if os.path.getsize(face_path) < 1e6:
    face_path = None

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, onnxruntime as ort

class SRVGGNetCompact(nn.Module):  # Real-ESRGAN general-x4v3 architecture (BSD-3-Clause)
    def __init__(self, num_in_ch=3, num_out_ch=3, num_feat=64, num_conv=32, upscale=4):
        super().__init__()
        self.upscale = upscale
        self.body = nn.ModuleList([nn.Conv2d(num_in_ch, num_feat, 3, 1, 1), nn.PReLU(num_parameters=num_feat)])
        for _ in range(num_conv):
            self.body.append(nn.Conv2d(num_feat, num_feat, 3, 1, 1))
            self.body.append(nn.PReLU(num_parameters=num_feat))
        self.body.append(nn.Conv2d(num_feat, num_out_ch * upscale * upscale, 3, 1, 1))
        self.upsampler = nn.PixelShuffle(upscale)
    def forward(self, x):
        out = x
        for layer in self.body:
            out = layer(out)
        return self.upsampler(out) + F.interpolate(x, scale_factor=self.upscale, mode='nearest')

def export(model, size, out_path, lo, hi, atol):
    dummy = torch.zeros(1, 3, size, size)
    kw = dict(input_names=['input'], output_names=['output'], opset_version=17, do_constant_folding=True)
    try:
        torch.onnx.export(model, dummy, out_path, dynamo=False, **kw)
    except TypeError:
        torch.onnx.export(model, dummy, out_path, **kw)
    x = torch.rand(1, 3, size, size) * (hi - lo) + lo
    with torch.no_grad():
        ref = model(x).numpy()
    sess = ort.InferenceSession(out_path, providers=['CPUExecutionProvider'])
    out = sess.run(None, {sess.get_inputs()[0].name: x.numpy()})[0]
    err = float(np.abs(ref - out).max())
    print(out_path, 'max diff vs PyTorch =', round(err, 5))
    assert np.isfinite(out).all() and err < atol, 'verification failed'

sd = torch.load(sr_path, map_location='cpu')
sd = sd.get('params_ema', sd.get('params', sd))
sr = SRVGGNetCompact(); sr.load_state_dict(sd, strict=True); sr.eval()
export(sr, 128, 'realesr_general_x4v3.onnx', 0.0, 1.0, 5e-3)

In [ ]:
if face_path:
    import sys, types, os, shutil, subprocess
    # GFPGAN ki sirf 2 architecture files lete hain; basicsr ke 2 chhote hisse khud bana dete hain.
    if not os.path.exists('GFPGAN_repo'):
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/TencentARC/GFPGAN', 'GFPGAN_repo'], check=True)
    os.makedirs('gfp_arch', exist_ok=True)
    for f in ['gfpganv1_clean_arch.py', 'stylegan2_clean_arch.py']:
        shutil.copy('GFPGAN_repo/gfpgan/archs/' + f, 'gfp_arch/' + f)
    open('gfp_arch/__init__.py', 'w').close()

    class _Registry:
        def register(self, *a, **k):
            return lambda cls: cls
    for name in ['basicsr', 'basicsr.utils', 'basicsr.utils.registry', 'basicsr.archs', 'basicsr.archs.arch_util']:
        sys.modules[name] = types.ModuleType(name)
    sys.modules['basicsr.utils.registry'].ARCH_REGISTRY = _Registry()
    sys.modules['basicsr.archs.arch_util'].default_init_weights = lambda *a, **k: None  # weights are loaded anyway

    sys.path.insert(0, '.')
    from gfp_arch.gfpganv1_clean_arch import GFPGANv1Clean
    net = GFPGANv1Clean(out_size=512, num_style_feat=512, channel_multiplier=2, decoder_load_path=None,
                        fix_decoder=False, num_mlp=8, input_is_latent=True, different_w=True, narrow=1, sft_half=True)
    sd = torch.load(face_path, map_location='cpu')
    net.load_state_dict(sd.get('params_ema', sd.get('params', sd)), strict=True); net.eval()
    class Wrap(nn.Module):
        def __init__(self, n): super().__init__(); self.n = n
        def forward(self, x): return self.n(x, return_rgb=False, randomize_noise=False)[0]
    export(Wrap(net).eval(), 512, 'gfpgan_v1_4.onnx', -1.0, 1.0, 5e-2)
else:
    print('GFPGAN file nahi di, face model skip.')

In [ ]:
import os, shutil
from google.colab import drive
drive.mount('/content/drive')
out = '/content/drive/MyDrive/PureEnhance_models'
os.makedirs(out, exist_ok=True)
for f in ['realesr_general_x4v3.onnx', 'gfpgan_v1_4.onnx']:
    if os.path.exists(f):
        shutil.copy(f, out)
        print('saved to Drive:', f, round(os.path.getsize(f) / 1e6, 1), 'MB')
print('Ab Google Drive app mein PureEnhance_models folder kholein.')